<a href="https://colab.research.google.com/github/Chen-M-22/GridCast/blob/main/05_final_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import keras
from keras import layers
import matplotlib.pyplot as plt

In [2]:
historical_demand_df = pd.read_csv('ercot_hourly_demand_2020_2025_clean.csv', parse_dates=['timestamp'])

In [3]:
historical_demand_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52608 entries, 0 to 52607
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   timestamp  52608 non-null  datetime64[ns]
 1   demand     52608 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 822.1 KB


In [4]:
historical_demand_df.isna().sum()

,0
timestamp,0
demand,0


In [5]:
historical_demand_df['timestamp'].duplicated().sum()

np.int64(0)

In [6]:
historical_demand_df.head()

,timestamp,demand
0,2020-01-01 00:00:00,40020.0
1,2020-01-01 01:00:00,41642.0
2,2020-01-01 02:00:00,40925.0
3,2020-01-01 03:00:00,39980.0
4,2020-01-01 04:00:00,39165.0


In [7]:
historical_demand_df.tail()

,timestamp,demand
52603,2025-12-31 19:00:00,49656.0
52604,2025-12-31 20:00:00,48726.0
52605,2025-12-31 21:00:00,48059.0
52606,2025-12-31 22:00:00,48037.0
52607,2025-12-31 23:00:00,48403.0


In [8]:
historical_demand_df['timestamp'].is_monotonic_increasing

True

Time Series Data Generation and Data Split

In [9]:
input_length = 168
horizon = 24

x = []
y = []

target_start_times = []
target_end_times = []

for i in range (input_length, len(historical_demand_df) - horizon + 1):
  x.append(historical_demand_df['demand'].iloc[i-input_length: i].to_numpy())
  y.append(historical_demand_df['demand'].iloc[i: i + horizon].to_numpy())

  target_start_times.append(historical_demand_df['timestamp'].iloc[i])
  target_end_times.append(historical_demand_df['timestamp'].iloc[i + horizon - 1])

target_start_times = pd.Series(target_start_times)
target_end_times = pd.Series(target_end_times)

train_mask = target_end_times < pd.Timestamp('2024-01-01')
val_mask = (target_start_times >= pd.Timestamp('2024-01-01')) & (target_end_times < pd.Timestamp('2025-01-01'))
test_mask = (target_start_times >= pd.Timestamp('2025-01-01'))

x = np.array(x)
y = np.array(y)

x_train = x[train_mask]
y_train = y[train_mask]

x_val = x[val_mask]
y_val = y[val_mask]

x_test = x[test_mask]
y_test = y[test_mask]

In [10]:
print('Total Windows:', len(x))
print('Total Train:', train_mask.sum())
print('Total Validation:', val_mask.sum())
print('Total Test:', test_mask.sum())
print('Excluded:', len(x) - train_mask.sum() - val_mask.sum() - test_mask.sum())
print('\n')
print('Train Dataset:', x_train.shape, y_train.shape)
print('Validation Dataset:', x_val.shape, y_val.shape)
print('Test Dataset:', x_test.shape, y_test.shape)

Total Windows: 52417
Total Train: 34873
Total Validation: 8761
Total Test: 8737
Excluded: 46


Train Dataset: (34873, 168) (34873, 24)
Validation Dataset: (8761, 168) (8761, 24)
Test Dataset: (8737, 168) (8737, 24)


##Data Standardization

In [11]:
train_demand = historical_demand_df[historical_demand_df['timestamp'] < pd.Timestamp('2024-01-01')]['demand']
train_mean = train_demand.mean()
train_std = train_demand.std()

x_train_scaled = (x_train - train_mean) / train_std
y_train_scaled = (y_train - train_mean) / train_std

x_val_scaled = (x_val - train_mean) / train_std
y_val_scaled = (y_val - train_mean) / train_std

x_test_scaled = (x_test - train_mean) / train_std
y_test_scaled = (y_test - train_mean) / train_std

In [12]:
len(train_demand)
print('training data mean:', train_mean)
print('training data standard deviation:', train_std)
print('\n')

print(x_train_scaled.shape, y_train_scaled.shape)
print(x_val_scaled.shape, y_val_scaled.shape)
print(x_test_scaled.shape, y_test_scaled.shape)

print(np.isnan(x_train_scaled).sum())
print(np.isinf(x_train_scaled).sum())

print(np.isnan(x_test_scaled).sum())
print(np.isinf(x_test_scaled).sum())

training data mean: 47083.72900981063
training data standard deviation: 10754.099125300023


(34873, 168) (34873, 24)
(8761, 168) (8761, 24)
(8737, 168) (8737, 24)
0
0
0
0


In [13]:
# Convert input data to (samples, timesteps, features).
# x has one feature, demand.

x_train_models = x_train_scaled[:, :, np.newaxis]
x_val_models = x_val_scaled[:, :, np.newaxis]
x_test_models = x_test_scaled[:, :, np.newaxis]

## Load Trained Models and Generate Test Predictions

In [15]:
lstm_model = keras.models.load_model('gridcast_lstm_v1.keras')

In [33]:
class PositionEmbedding(keras.Layer):
  def __init__(self, sequence_length, output_dim,  **kwargs):
    super().__init__(**kwargs)

    self.sequence_length = sequence_length
    self.output_dim = output_dim

    self.position_embeddings = layers.Embedding(input_dim = sequence_length, output_dim = output_dim)

  def build(self, input_shape):
      self.position_embeddings.build((None,))
      super().build(input_shape)

  def call(self, inputs):
    positions = keras.ops.arange(0, keras.ops.shape(inputs)[1])
    embedded_positions = self.position_embeddings(positions)
    return embedded_positions

  def get_config(self):
    config = super().get_config()
    config.update({
        "sequence_length": self.sequence_length,
        "output_dim": self.output_dim
    })
    return config

In [34]:
transformer_model = keras.models.load_model('gridcast_transformer_v1.keras', custom_objects={'PositionEmbedding': PositionEmbedding}, compile=False)

In [35]:
lstm_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 168, 1)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 55,370 (216.29 KB)

 Trainable params: 18,456 (72.09 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 36,914 (144.20 KB)

In [36]:
transformer_model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7       │ (None, 168, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_25 (Dense)    │ (None, 168, 32)   │         64 │ input_layer_7[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ position_embedding… │ (168, 32)         │      5,376 │ dense_25[0][0]    │
│ (PositionEmbedding) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_17 (Add)        │ (None, 168, 32)   │          0 │ dense_25[0][0],   │
│                     │                   │            │ position_embeddi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 168, 32)   │      4,224 │ add_17[0][0],     │
│ (MultiHeadAttentio… │                   │            │ add_17[0][0],     │
│                     │                   │            │ add_17[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_18 (Add)        │ (None, 168, 32)   │          0 │ add_17[0][0],     │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 168, 32)   │         64 │ add_18[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_26 (Dense)    │ (None, 168, 64)   │      2,112 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_27 (Dense)    │ (None, 168, 32)   │      2,080 │ dense_26[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_19 (Add)        │ (None, 168, 32)   │          0 │ layer_normalizat… │
│                     │                   │            │ dense_27[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 168, 32)   │         64 │ add_19[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_6          │ (None, 32)        │          0 │ layer_normalizat… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_28 (Dense)    │ (None, 24)        │        792 │ get_item_6[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 14,776 (57.72 KB)

 Trainable params: 14,776 (57.72 KB)

 Non-trainable params: 0 (0.00 B)

In [37]:
y_test_pred_lstm_scaled = lstm_model(x_test_models, training = False)
y_test_pred_lstm = y_test_pred_lstm_scaled.numpy() * train_std + train_mean

In [38]:
test_lstm_mae = np.abs(y_test_pred_lstm - y_test).mean()
test_lstm_rmse = np.sqrt(np.square(y_test_pred_lstm - y_test).mean())
normalized_test_lstm_mae = (test_lstm_mae / y_test.mean()) * 100

In [39]:
print('Test MAE:', test_lstm_mae)
print('Test RMSE:', test_lstm_rmse)
print('Normalized Test MAE:', normalized_test_lstm_mae)

Test MAE: 2268.1851188437768
Test RMSE: 3246.5004667328717
Normalized Test MAE: 4.069953277983815


In [40]:
y_test_pred_transformer_scaled = transformer_model(x_test_models, training = False)
y_test_pred_transformer = y_test_pred_transformer_scaled.numpy() * train_std + train_mean

In [41]:
test_transformer_mae = np.abs(y_test_pred_transformer - y_test).mean()
test_transformer_rmse = np.sqrt(np.square(y_test_pred_transformer - y_test).mean())
normalized_test_transformer_mae = (test_transformer_mae / y_test.mean()) * 100

In [42]:
print('Test MAE:', test_transformer_mae)
print('Test RMSE:', test_transformer_rmse)
print('Normalized Test MAE:', normalized_test_transformer_mae)

Test MAE: 2073.062535498681
Test RMSE: 2958.230300832275
Normalized Test MAE: 3.719832032986466


In [43]:
# Daily Seaonality Model

y_test_pred_daily = x_test[:, -24: ]

test_daily_mae = np.abs(y_test_pred_daily - y_test).mean()
test_daily_rmse = np.sqrt(np.square(y_test_pred_daily - y_test).mean())
normalized_test_daily_mae = (test_daily_mae / y_test.mean()) * 100

In [44]:
print('Test MAE:', test_daily_mae)
print('Test RMSE:', test_daily_rmse)
print('Normalized Test MAE:', normalized_test_daily_mae)

Test MAE: 2522.3258269428866
Test RMSE: 3606.545765480325
Normalized Test MAE: 4.525974613898566


In [45]:
test_results = pd.DataFrame({
    "Model": [
        "Daily Seasonal Baseline",
        "LSTM",
        "Transformer"
    ],
    "MAE (MW)": [
        test_daily_mae,
        test_lstm_mae,
        test_transformer_mae
    ],
    "RMSE (MW)": [
        test_daily_rmse,
        test_lstm_rmse,
        test_transformer_rmse
    ],
    "Normalized MAE (%)": [
        normalized_test_daily_mae,
        normalized_test_lstm_mae,
        normalized_test_transformer_mae
    ]
})

test_results

,Model,MAE (MW),RMSE (MW),Normalized MAE (%)
0,Daily Seasonal Baseline,2522.325827,3606.545765,4.525975
1,LSTM,2268.185119,3246.500467,4.069953
2,Transformer,2073.062535,2958.230301,3.719832


## Final Test Results
On the previously untouched 2025 test set, both deep-learning models outperformed the daily seasonal baseline. The LSTM achieved a test MAE of approximately 2,268 MW, while the Transformer achieved the best overall performance with a test MAE of approximately 2,073 MW. Relative to the daily seasonal baseline, the Transformer reduced MAE by approximately 17.8%. It also outperformed the LSTM by approximately 8.6% in MAE.

These results indicate that information contained in the previous 168 hours of electricity demand can improve 24-hour-ahead forecasting beyond a simple daily seasonal forecast. Among the models evaluated in this project, the Transformer generalized best to the unseen 2025 test period.